Implementation of Metropolis adjusted IPLA (in a MALA form) for Bayesian Logistic regression

In [ ]:
from pathlib import Path
import numpy as np 
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

In [ ]:
data = np.load("../data/synthetic_logistic_regression.npz")
sigma = 0.05

v = jnp.asarray(data["v"])
y = jnp.asarray(data["y"])
x_true = jnp.asarray(data["x_true"])

In [ ]:
def sumU(theta, X, v=v, y=y, sigma=sigma):
    log_lik = jnp.sum(y * jax.nn.log_sigmoid(X @ v.T) 
                         + (1-y) * jax.nn.log_sigmoid(-X @ v.T))
    log_prior = - 0.5 / sigma**2 * (jnp.log(2 * jnp.pi * sigma**2)) 
    + jnp.sum((X-theta)**2) /sigma**2

    return - jnp.sum(log_lik + log_prior)

Ugrad_th = jax.grad(sumU, argnums=0)
Ugrad_X = jax.grad(sumU, argnums=1)

def log_phi(theta_prop, theta, X, v, y, gamma, key):
    N = X.shape[0]
    mean = theta - gamma / N * Ugrad_th(theta, X)
    var = 2*gamma/N
    phi = - theta.shape[0] / 2 * jax.log(2*jax.pi*sigma**2)
    - 0.5 * jnp.sum((theta_prop - mean)**2) / var

    return phi